## Initialization

In [ ]:
import os, json, pickle
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import joblib

from torch_geometric.data import Data
from torch_geometric.nn import MessagePassing
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score

city = "Englewood"
train_storms = ['2Y6h','10Y6h','100Y6h','500Y6h','2Y_100Y6h','100Y_2Y6h']
eval_storms  = ['5Y6h','25Y6h','50Y6h','200Y6h','1000Y6h']
unit = 'ft' # 'ft' or 'm'

data_path = os.path.join(os.getcwd(), "data")
save_dir  = os.path.join(data_path, f"artifacts_{city}")
device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")

d = torch.load(os.path.join(save_dir, "pyg_tensors.pt"), map_location=device)
data = Data(x=d["x"], edge_index=d["edge_index"])
for k in ["edge_attr", "y", "node_weight", "pos"]:
    if d.get(k) is not None:
        setattr(data, k, d[k])

with open(os.path.join(save_dir, "nx_graph.pkl"), "rb") as f:
    G = pickle.load(f)

with open(os.path.join(save_dir, "meta.json")) as f:
    meta = json.load(f)

FEAT_KEYS  = meta["feat_keys"]
TARGET_KEY = meta["target_key"]
storm_cols = meta["storm_cols"]

coords      = np.load(os.path.join(save_dir, "coords.npy"))
node_weight = np.load(os.path.join(save_dir, "node_weight.npy"))

target_depths = {
    k: v for k, v in np.load(os.path.join(save_dir, "depth.npz")).items()
}

rainfall = pd.read_csv(os.path.join(data_path, "Rainfall.csv"))
for c in storm_cols:
    if c not in rainfall.columns:
        raise KeyError(f"Rainfall.csv missing column '{c}' expected from meta['storm_cols']")
    rainfall[c] = pd.to_numeric(rainfall[c], errors="coerce")

rain_series   = {}
storm_lengths = {}
for c in storm_cols:
    vals = rainfall[c].dropna().to_numpy(dtype=np.float32)
    storm_lengths[c] = len(vals)
    rain_series[c] = torch.tensor(vals, dtype=torch.float32).view(-1, 1)

print("\n=== Rainfall sequence lengths (after dropping NaNs) ===")
for c in storm_cols:
    print(f"{c}: T = {storm_lengths[c]}")

print("\n", data)
print(f"Loaded NX graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

high_occ_mask = node_weight > 1
print(f"High-occupancy nodes: {int(high_occ_mask.sum())} / {len(node_weight)}")

## Model Definition

In [ ]:
class MPEdgeConv(MessagePassing):
    def __init__(self, in_channels, edge_dim, out_channels, edge_hidden=8):
        super().__init__(aggr="max")
        self.edge_mlp = nn.Sequential(
            nn.Linear(edge_dim, edge_hidden),
            nn.ReLU(),
            nn.Linear(edge_hidden, edge_hidden),
        )
        self.mlp = nn.Sequential(
            nn.Linear(2 * in_channels + edge_hidden, out_channels),
            nn.ReLU(),
            nn.Linear(out_channels, out_channels),
        )

    def forward(self, x, edge_index, edge_attr):
        return self.propagate(edge_index, x=x, edge_attr=edge_attr)

    def message(self, x_i, x_j, edge_attr):
        diff = x_j - x_i
        e = self.edge_mlp(edge_attr)
        z = torch.cat([x_i, diff, e], dim=-1)
        return self.mlp(z)


class FloodMAGNet(nn.Module):
    def __init__(self, in_dim, edge_dim, rain_hidden=16, gnn_hidden=64, edge_hidden=8):
        super().__init__()
        self.rain_embed = nn.Linear(1, rain_hidden)
        self.rain_attn  = nn.Linear(rain_hidden, 1)

        self.conv1 = MPEdgeConv(in_dim,       edge_dim, gnn_hidden, edge_hidden=edge_hidden)
        self.conv2 = MPEdgeConv(gnn_hidden,   edge_dim, gnn_hidden, edge_hidden=edge_hidden)
        self.conv3 = MPEdgeConv(gnn_hidden,   edge_dim, gnn_hidden, edge_hidden=edge_hidden)
        self.conv4 = MPEdgeConv(gnn_hidden + rain_hidden, edge_dim, gnn_hidden, edge_hidden=edge_hidden)
        self.conv5 = MPEdgeConv(gnn_hidden,   edge_dim, gnn_hidden, edge_hidden=edge_hidden)
        self.conv6 = MPEdgeConv(gnn_hidden,   edge_dim, gnn_hidden, edge_hidden=edge_hidden)

        self.out = nn.Linear(gnn_hidden, 1)

    def forward(self, data, rain_seq):
        if rain_seq.dim() == 1:
            rain_seq = rain_seq.view(-1, 1)

        h = torch.tanh(self.rain_embed(rain_seq))
        scores = self.rain_attn(h).squeeze(-1)
        alpha = torch.softmax(scores, dim=0)

        rain_feat = torch.sum(alpha.unsqueeze(-1) * h, dim=0, keepdim=True)
        rain_broadcast = rain_feat.expand(data.num_nodes, -1)

        edge_attr  = data.edge_attr[:, [0, 2, 3]]
        edge_index = data.edge_index

        x1 = F.relu(self.conv1(data.x, edge_index, edge_attr))
        x2 = F.relu(self.conv2(x1, edge_index, edge_attr)) + x1
        x3 = F.relu(self.conv3(x2, edge_index, edge_attr)) + x2

        x3 = torch.cat([x3, rain_broadcast], dim=1)

        x4 = F.relu(self.conv4(x3, edge_index, edge_attr))
        x5 = F.relu(self.conv5(x4, edge_index, edge_attr)) + x4
        x6 = F.relu(self.conv6(x5, edge_index, edge_attr)) + x5

        yhat = self.out(x6).squeeze(1)
        yhat = F.softplus(yhat)
        return yhat

### True Depth Function & Importance Loss Definition

In [ ]:
def get_y_true(storm, device):
    y = np.asarray(target_depths[storm], dtype=float)
    y = np.nan_to_num(y, nan=0.0).astype(np.float32)
    return torch.tensor(y, dtype=torch.float32, device=device)

def importance_weighted_mse(pred, target, node_weight):
    node_weight = node_weight.to(pred.device).float()
    node_weight = node_weight / node_weight.mean()
    with torch.no_grad():
        error = (pred - target).abs()
        if unit == 'm':
            target = target * 3.28084
            error = error * 3.28084
        ## Critical weight parameters
        alpha = 1.5
        beta = 0.8
        theta = 1
        gamma = 4
        omega = 8

        critical_weight_depth = 1 + alpha * torch.exp(-theta * target)
        critical_weight_error = torch.clamp(torch.exp(beta * error), 1.0, gamma)
        critical_weight = critical_weight_depth * critical_weight_error
        critical_weight = critical_weight.clamp(min=1, max=omega)
    weights = node_weight * critical_weight
    return (weights * (pred - target) ** 2).mean()

## Training

In [ ]:
import random
import warnings
warnings.filterwarnings("ignore")

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

model = FloodMAGNet(
    in_dim=data.x.size(1),
    edge_dim=3,
    rain_hidden=16,
    gnn_hidden=64,
    edge_hidden=8,
).to(device)

opt  = torch.optim.Adam(model.parameters(), lr=3e-3)
data = data.to(device)

EPOCHS   = 2000
VAL_EVERY = 20

best_val   = float("inf")
best_epoch = -1

train_losses = []
val_losses   = []

ckpt_path = os.path.join(save_dir, "floodmagnet_best_test.pt")

for epoch in range(1, EPOCHS + 1):
    model.train()
    random.shuffle(train_storms)

    running = 0.0
    for storm in train_storms:
        rain   = rain_series[storm].to(device)
        y_true = get_y_true(storm, device=device)
        y_pred = model(data, rain)

        loss = importance_weighted_mse(y_pred, y_true, data.node_weight)

        opt.zero_grad()
        loss.backward()
        opt.step()

        running += loss.item()
    print(f"Epoch {epoch}/{EPOCHS}, IWMSE = {loss}", end='\r')
    train_losses.append(running / len(train_storms))

    if epoch % VAL_EVERY == 0:
        model.eval()
        val_total = 0.0

        with torch.no_grad():
            for storm in eval_storms:
                rain   = rain_series[storm].to(device)
                y_true = get_y_true(storm, device=device)
                y_pred = model(data, rain)
                val_total += importance_weighted_mse(y_pred, y_true, data.node_weight).item()

        avg_val = val_total / len(eval_storms)
        val_losses.append(avg_val)

        if avg_val < best_val:
            best_val = avg_val
            best_epoch = epoch
            torch.save(
                {
                    "epoch": epoch,
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": opt.state_dict(),
                    "best_val_iwmse": best_val,
                    "train_storms": train_storms,
                    "eval_storms": eval_storms,
                },
                ckpt_path,
            )
            print(f"Best at epoch {epoch} ; Val IWMSE: {best_val:.6f}")

    if epoch % 50 == 0:
        print(f"Epoch {epoch}/{EPOCHS} ; Train IWMSE: {train_losses[-1]:.6f}")

if os.path.exists(ckpt_path):
    ckpt = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(ckpt["model_state_dict"])
    print(f"\nBest model from epoch {ckpt['epoch']} with Val IWMSE={ckpt['best_val_iwmse']:.6f}")


## Evaluation

### Helpers

In [ ]:
def compute_nse(true, pred):
    return 1 - np.sum((true - pred) ** 2) / np.sum((true - np.mean(true)) ** 2)

def compute_kge(true, pred):
    r = np.corrcoef(true, pred)[0, 1]
    alpha = np.std(pred) / np.std(true)
    beta = np.mean(pred) / np.mean(true)
    return 1 - np.sqrt((r - 1) ** 2 + (alpha - 1) ** 2 + (beta - 1) ** 2)

def compute_wrmse(true, pred, weights):
    return np.sqrt(np.average((true - pred) ** 2, weights=weights))

def get_metrics(y_true, y_pred, weights):
    rmse  = root_mean_squared_error(y_true, y_pred)
    mae   = mean_absolute_error(y_true, y_pred)
    wmape = np.abs(y_true - y_pred).sum() / np.maximum(y_true.sum(), 1e-8) * 100
    nse   = compute_nse(y_true, y_pred)
    kge   = compute_kge(y_true, y_pred)
    wrmse = compute_wrmse(y_true, y_pred, weights)
    r2    = r2_score(y_true, y_pred)
    return rmse, mae, wmape, nse, kge, wrmse, r2

### Evaluation over all test storms

In [ ]:
rmse_all, mae_all, wmape_all, nse_all, kge_all, wrmse_all, r2_all = [], [], [], [], [], [], []
rmse_hi,  mae_hi,  wmape_hi,  nse_hi,  kge_hi,  wrmse_hi,  r2_hi  = [], [], [], [], [], [], []

for storm in eval_storms:
    model.eval()
    with torch.no_grad():
        pred = model(data, rain_series[storm].to(device)).cpu().numpy().ravel()

    for i, v in enumerate(pred):
        G.nodes[i][f"pred_depth_{storm}_{unit}"] = float(v)

    true = np.asarray(target_depths[storm], dtype=float)
    w    = np.asarray(node_weight, dtype=float)

    mask = np.isfinite(true) & np.isfinite(pred)
    true_m = true[mask]
    pred_m = pred[mask]
    w_m    = w[mask]

    rmse, mae, wmape, nse, kge, wrmse, r2 = get_metrics(true_m, pred_m, w_m)
    rmse_all.append(rmse); mae_all.append(mae); wmape_all.append(wmape)
    nse_all.append(nse); kge_all.append(kge); wrmse_all.append(wrmse); r2_all.append(r2)

    hi = w_m > 1
    rmse, mae, wmape, nse, kge, wrmse, r2 = get_metrics(true_m[hi], pred_m[hi], w_m[hi])
    rmse_hi.append(rmse); mae_hi.append(mae); wmape_hi.append(wmape)
    nse_hi.append(nse); kge_hi.append(kge); wrmse_hi.append(wrmse); r2_hi.append(r2)

    r2_hex = r2_score(true_m, pred_m)

    plt.figure(figsize=(7, 7))
    hb = plt.hexbin(true_m, pred_m, gridsize=200, bins="log")
    plt.plot([0, true_m.max()], [0, true_m.max()], "r--", label="y = x")
    plt.colorbar(hb, label="Point density (log scale)")
    plt.text(0.05 * true_m.max(), 0.9 * true_m.max(), f"$R^2$ = {r2_hex:.3f}", fontsize=14)
    plt.xlabel(f"True Depth ({unit})")
    plt.ylabel(f"Predicted Depth ({unit})")
    plt.title(f"True vs Predicted Depths ({storm})")
    plt.legend()
    plt.show()

    true_hi = true_m[hi]
    pred_hi = pred_m[hi]
    r2_hex_hi = r2_score(true_hi, pred_hi) if len(true_hi) > 1 else np.nan

    plt.figure(figsize=(7, 7))
    hb = plt.hexbin(true_hi, pred_hi, gridsize=200, bins="log")
    if np.isfinite(true_hi).any():
        plt.plot([0, true_hi.max()], [0, true_hi.max()], "r--", label="y = x")
    plt.colorbar(hb, label="Point density (log scale)")
    if np.isfinite(r2_hex_hi):
        plt.text(0.05 * true_hi.max(), 0.9 * true_hi.max(), f"$R^2$ = {r2_hex_hi:.3f}", fontsize=14)
    plt.xlabel(f"True Depth ({unit})")
    plt.ylabel(f"Predicted Depth ({unit})")
    plt.title(f"True vs Predicted Depths ({storm}) — High-Risk Nodes")
    plt.legend()
    plt.show()

    plt.figure(figsize=(10, 10))
    sc = plt.scatter(coords[:, 0], coords[:, 1], c=pred, s=10, edgecolor="k", linewidth=0.1, vmax=3)
    plt.colorbar(sc, label=f"Predicted Flood Depth ({unit})")
    plt.title(f"Predicted {storm} Flood Depth ({unit})")
    plt.axis("off")
    plt.axis("equal")
    plt.show()

    plt.figure(figsize=(10, 10))
    sc = plt.scatter(coords[:, 0], coords[:, 1], c=true, s=10, edgecolor="k", linewidth=0.0, vmax=3)
    plt.colorbar(sc, label=f"Simulated Flood Depth ({unit})")
    plt.title(f"Simulated {storm} Flood Depth ({unit})")
    plt.axis("off")
    plt.axis("equal")
    plt.show()

    denom = np.maximum(np.abs(true), 1e-3)
    err_pct = np.abs(pred - true) / denom * 100.0
    err_pct[~np.isfinite(err_pct)] = np.nan

    plt.figure(figsize=(10, 10))
    sc = plt.scatter(coords[:, 0], coords[:, 1], c=err_pct, s=10, edgecolor="k", linewidth=0.0, vmin=0, vmax=100)
    plt.colorbar(sc, label="Absolute Error (%)")
    plt.title(f"Absolute Percent Error Map ({storm})")
    plt.axis("off")
    plt.axis("equal")
    plt.show()


print(f"\nMetrics for all nodes, averaged over {eval_storms}")
print(f"  RMSE  : {np.mean(rmse_all):.3f} {unit}")
print(f"  MAE   : {np.mean(mae_all):.3f} {unit}")
print(f"  WMAPE : {np.mean(wmape_all):.2f}%")
print(f"  NSE   : {np.mean(nse_all):.3f}")
print(f"  KGE   : {np.mean(kge_all):.3f}")
print(f"  WRMSE : {np.mean(wrmse_all):.3f} {unit}")
print(f"  R2    : {np.mean(r2_all):.3f}")

print(f"\nMetrics for high risk nodes, averaged over {eval_storms}")
print(f"  RMSE  : {np.mean(rmse_hi):.3f} {unit}")
print(f"  MAE   : {np.mean(mae_hi):.3f} {unit}")
print(f"  WMAPE : {np.mean(wmape_hi):.2f}%")
print(f"  NSE   : {np.mean(nse_hi):.3f}")
print(f"  KGE   : {np.mean(kge_hi):.3f}")
print(f"  WRMSE : {np.mean(wrmse_hi):.3f} {unit}")
print(f"  R2    : {np.mean(r2_hi):.3f}")